In [2]:
import torch 
import numpy as np
from pathlib import Path
import json
from skills_utility.skills_match import (
    Jobseeker,
    Opportunity,
    SimilarityEngine,
    compute_U_complete,
)



MODEL_PATH =  "skills_utility/Output/skill_embedding_model.pt"
state = torch.load(MODEL_PATH, map_location="cpu")
W = state["state_dict"]["embedding.weight"].numpy()
norms = np.linalg.norm(W, axis=1, keepdims=True)
W = W / np.where(norms > 0, norms, 1.0)
MAPPING_PATH = "skills_utility/Output/skill_to_row.json"
with open(MAPPING_PATH, "r") as f:
    skill_to_row = json.load(f)




C:\Users\Yared\AppData\Local\Temp\ipykernel_13364\3740962595.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(MODEL_PATH, map_location="cpu")


In [6]:
engine = SimilarityEngine(W, skill_to_row)
js_ids = list(['b964b975-329d-468d-b5e2-f09c3b9fdbfc',]) # "paskal"
ess_ids = list(['68933878ba407998dac8c748','68933878ba407998dac8c747']) # "court procedures",
js_mat, js_ids_valid = engine._rows_with_ids(js_ids)
ess_mat, ess_ids_valid = engine._rows_with_ids(ess_ids)
# 3. Essential Skill Similarity (Best-of Cosine)
ess_sim, ess_rowmax = engine.best_mean_cos(ess_mat, js_mat)
ess_sim,ess_rowmax

(0.2755497694015503, array([0.55109954, 0.        ], dtype=float32))

In [21]:
S = ess_mat @ js_mat.T
np.maximum(S, 0.0, out=S) # Clip negative similarities
rowmax = S.max(axis=1)    # Max similarity for each job skill
float(rowmax.mean()), rowmax

(0.0, array([0.], dtype=float32))

In [24]:
ess_mat@js_mat.T

array([[-0.04753056]], dtype=float32)

In [ ]:
import os
import json
import random
import pandas as pd
import networkx as nx
import torch

from gensim.models import Word2Vec

# =============================================================================
# 1) DATA LOADING
# =============================================================================

def load_taxonomy(taxonomy_path: str):
    """Reads taxonomy CSVs from the specified directory."""
    skills = pd.read_csv(os.path.join(taxonomy_path, 'skills.csv'))
    hierarchy = pd.read_csv(os.path.join(taxonomy_path, 'skill_hierarchy.csv'))
    
    # Safely handle the optional relations file if it doesn't exist
    rel_path = os.path.join(taxonomy_path, 'skill_to_skill_relations.csv')
    if os.path.exists(rel_path):
        relations = pd.read_csv(rel_path)
    else:
        relations = pd.DataFrame(columns=['REQUIRINGID', 'REQUIREDID'])
    return skills, hierarchy, relations

# =============================================================================
# 2) GRAPH CONSTRUCTION
# =============================================================================

def build_unweighted_skill_graph(
    skills_df,
    hierarchy_df,
    relations_df
):
    """
    Build graph and save simple adjacency
    for debugging random walks.
    """

    G = nx.Graph()

    # =========================================
    # CREATE NODE LIST
    # =========================================

    node_list = [
        str(x)
        for x in skills_df["ID"]
        .dropna()
        .unique()
        .tolist()
    ]

    node_to_idx = {
        node_id: i
        for i, node_id in enumerate(node_list)
    }

    # =========================================
    # ADD NODES
    # =========================================

    for node_id in node_list:
        G.add_node(node_id)

    # =========================================
    # ADD HIERARCHY EDGES
    # =========================================

    for _, row in hierarchy_df.iterrows():

        parent_id = str(
            row.get("PARENTID")
        )

        child_id = str(
            row.get("CHILDID")
        )

        if parent_id in G and child_id in G:
            G.add_edge(parent_id, child_id)
        else:
            #fetch all children from parents
            

    # =========================================
    # ADD RELATION EDGES
    # =========================================

    for _, row in relations_df.iterrows():

        requiring_id = str(
            row.get("REQUIRINGID")
        )

        required_id = str(
            row.get("REQUIREDID")
        )

        if requiring_id in G and required_id in G:
            G.add_edge(requiring_id, required_id)

    # =========================================
    # SAVE SIMPLE ADJACENCY JSON
    # =========================================

    adjacency = {}

    for node in G.nodes():

        adjacency[node] = list(
            G.neighbors(node)
        )

    with open(
        "skills_utility/Output/graph_adjacency.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            adjacency,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        "Saved adjacency to "
        "graph_adjacency.json"
    )

    return G, node_to_idx, node_list

# =============================================================================
# 3) MODEL TRAINING (Node2Vec Logic via Gensim)
# =============================================================================

def train_and_save(graph, node_list, artifacts_path):

    # =====================================================
    # RANDOM WALK GENERATOR
    # =====================================================

    def generate_random_walks(
        G,
        num_walks=20,
        walk_length=30
    ):

        walks = []

        nodes = list(G.nodes())

        for _ in range(num_walks):

            random.shuffle(nodes)

            for node in nodes:

                walk = [node]

                while len(walk) < walk_length:

                    cur = walk[-1]

                    neighbors = list(
                        G.neighbors(cur)
                    )

                    if not neighbors:
                        break

                    next_node = random.choice(
                        neighbors
                    )

                    walk.append(next_node)

                walks.append(walk)

        return walks

    print("Step 1: Generating random walks...")

    walks = generate_random_walks(
        graph,
        num_walks=20,
        walk_length=30
    )

    # =====================================================
    # SAVE RANDOM WALKS
    # =====================================================

    os.makedirs(artifacts_path, exist_ok=True)

    # ---------- JSON ----------
    json_path = os.path.join(
        artifacts_path,
        "random_walks.json"
    )

    with open(json_path, "w") as f:
        json.dump(walks, f, indent=2)

    # ---------- TXT ----------
    txt_path = os.path.join(
        artifacts_path,
        "random_walks.txt"
    )

    with open(txt_path, "w") as f:
        for walk in walks:
            f.write(" ".join(map(str, walk)))
            f.write("\n")

    # ---------- CSV ----------
    csv_path = os.path.join(
        artifacts_path,
        "random_walks.csv"
    )

    pd.DataFrame({
        "walk": [
            " -> ".join(map(str, walk))
            for walk in walks
        ]
    }).to_csv(csv_path, index=False)

    print("Random walks saved.")

    print(
        f"Artifacts successfully saved to: "
        f"{artifacts_path}"
    )
    return walks
# =============================================================================
# 4) EXECUTION
# =============================================================================


script_dir = "skills_utility"
TAXONOMY_DIR = script_dir
OUTPUT_DIR = os.path.join(script_dir, "Output")

# Execute training pipeline
s_df, h_df, r_df = load_taxonomy(TAXONOMY_DIR)
G, n2idx, n_list = build_unweighted_skill_graph(s_df, h_df, r_df)
walks = train_and_save(G, n_list, OUTPUT_DIR)


Saved adjacency to graph_adjacency.json
Step 1: Generating random walks...
Random walks saved.
Artifacts successfully saved to: skills_utility\Output


In [29]:
import matplotlib.pyplot as plt
import networkx as nx


def visualize_walk(graph, walk):

    plt.figure(figsize=(12, 8))

    pos = nx.spring_layout(graph)
    print("--0")
    # draw full graph
    nx.draw(
        graph,
        pos,
        node_color="lightgray",
        edge_color="gray",
        with_labels=True,
        node_size=500
    )

    # highlight walk edges
    walk_edges = list(
        zip(walk[:-1], walk[1:])
    )
    print("--1")
    nx.draw_networkx_edges(
        graph,
        pos,
        edgelist=walk_edges,
        edge_color="red",
        width=3
    )

    nx.draw_networkx_nodes(
        graph,
        pos,
        nodelist=walk,
        node_color="orange",
        node_size=700
    )
    print('--2')

    plt.title("Random Walk Visualization")

    plt.show()

visualize_walk(G, walks[1])

KeyboardInterrupt: 

<Figure size 1200x800 with 0 Axes>

In [33]:
df_hierarchy = pd.read_csv("skills_utility/skill_hierarchy.csv")
df_hierarchy.head()

,PARENTOBJECTTYPE,PARENTID,CHILDID,CHILDOBJECTTYPE,CREATEDAT,UPDATEDAT
0,skillgroup,68933870ba407998dac8a1e5,68933870ba407998dac89fcc,skillgroup,2025-08-06T11:12:07.313Z,2025-08-06T11:12:07.313Z
1,skillgroup,68933870ba407998dac89fcc,68933870ba407998dac89fcd,skillgroup,2025-08-06T11:12:07.314Z,2025-08-06T11:12:07.314Z
2,skillgroup,68933870ba407998dac89fcd,68933870ba407998dac89fce,skillgroup,2025-08-06T11:12:07.314Z,2025-08-06T11:12:07.314Z
3,skillgroup,68933870ba407998dac89fcc,68933870ba407998dac89fcf,skillgroup,2025-08-06T11:12:07.314Z,2025-08-06T11:12:07.314Z
4,skillgroup,68933870ba407998dac89fcf,68933870ba407998dac89fd0,skillgroup,2025-08-06T11:12:07.314Z,2025-08-06T11:12:07.314Z


In [34]:
df_hierarchy[df_hierarchy['PARENTID']=='68933870ba407998dac89fd8'].head()

,PARENTOBJECTTYPE,PARENTID,CHILDID,CHILDOBJECTTYPE,CREATEDAT,UPDATEDAT
13,skillgroup,68933870ba407998dac89fd8,68933870ba407998dac89fd9,skillgroup,2025-08-06T11:12:07.314Z,2025-08-06T11:12:07.314Z
14,skillgroup,68933870ba407998dac89fd8,68933870ba407998dac89fda,skillgroup,2025-08-06T11:12:07.314Z,2025-08-06T11:12:07.314Z
15,skillgroup,68933870ba407998dac89fd8,68933870ba407998dac89fdb,skillgroup,2025-08-06T11:12:07.314Z,2025-08-06T11:12:07.314Z
16,skillgroup,68933870ba407998dac89fd8,68933870ba407998dac89fdc,skillgroup,2025-08-06T11:12:07.314Z,2025-08-06T11:12:07.314Z
17,skillgroup,68933870ba407998dac89fd8,68933870ba407998dac89fdd,skillgroup,2025-08-06T11:12:07.314Z,2025-08-06T11:12:07.314Z


In [36]:
df_hierarchy[df_hierarchy['PARENTID']=='68933870ba407998dac89fda'].head()

,PARENTOBJECTTYPE,PARENTID,CHILDID,CHILDOBJECTTYPE,CREATEDAT,UPDATEDAT
2578,skillgroup,68933870ba407998dac89fda,ea9c37df-3114-42b7-b19f-c34de6fc14cd,skill,2025-08-06T11:12:07.480Z,2025-08-06T11:12:07.480Z
3663,skillgroup,68933870ba407998dac89fda,1640b2e1-70fd-42da-83fa-913b8fbb754b,skill,2025-08-06T11:12:07.556Z,2025-08-06T11:12:07.556Z
4603,skillgroup,68933870ba407998dac89fda,8aada32c-15b4-406d-acce-e557df5e6fa4,skill,2025-08-06T11:12:07.598Z,2025-08-06T11:12:07.598Z
5937,skillgroup,68933870ba407998dac89fda,8b78d4b6-7d27-42a0-8388-110842489711,skill,2025-08-06T11:12:10.497Z,2025-08-06T11:12:10.497Z
6455,skillgroup,68933870ba407998dac89fda,4ca05222-f877-4be1-b3a4-d59f4824df90,skill,2025-08-06T11:12:10.536Z,2025-08-06T11:12:10.536Z
